Here is the blueprint we established, which remains exactly the same:

Module 1: The Data Loader & Formatter. * Module 2: The Splitter. * Module 3: The Preprocessing Pipeline. * Module 4: The Model & Evaluator.

In [47]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv(r'C:\Users\moham\Desktop\Data Science\BeCode\becode_projects\immo-eliza\data\processed\listings_with_distance_clean.csv')


In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23169 entries, 0 to 23168
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23169 non-null  str    
 1   region                  23169 non-null  str    
 2   zip_code                23169 non-null  int64  
 3   property_type           23169 non-null  str    
 4   subtype                 23169 non-null  str    
 5   price_eur               23169 non-null  float64
 6   type_of_sale            23169 non-null  str    
 7   num_rooms               23169 non-null  int64  
 8   living_area_m2          23169 non-null  float64
 9   fully_equipped_kitchen  23169 non-null  int64  
 10  furnished               23169 non-null  int64  
 11  terrace                 23169 non-null  int64  
 12  terrace_area_m2         15626 non-null  float64
 13  garden                  23169 non-null  int64  
 14  garden_area_m2          15292 non-null  float64
 

ELIMINATING OUTLIERS IN price_eur AND living_area_m2 TO SEE IF IT BETTERS THE LR SCORE

WE WILL USE IQR RULE:

The IQR Rule: > * Find the middle 50% (the "Box").

Anything further than 1.5 boxes away from the edges is an outlier.


In [50]:
# Calculate Q1, Q3 and IQR for price_eur
q1_price = df['price_eur'].quantile(0.25)
q3_price = df['price_eur'].quantile(0.75)
iqr_price = q3_price - q1_price

lower_price = q1_price - 1.5 * iqr_price
upper_price = q3_price + 1.5 * iqr_price

#Create the filter (keep rows where price is between the bounds)
df = df[(df['price_eur'] >= lower_price) & (df['price_eur'] <= upper_price)]


In [51]:
# Calculate Bounds for Area
q1_area = df['living_area_m2'].quantile(0.25)
q3_area = df['living_area_m2'].quantile(0.75)
iqr_area = q3_area - q1_area

lower_area = q1_area - 1.5 * iqr_area
upper_area = q3_area + 1.5 * iqr_area 

# Create the filter (keep rows where living area is between the bounds)
df = df[(df['living_area_m2'] >= lower_area) & (df['living_area_m2'] <= upper_area)]

In [52]:
df.shape

(20863, 22)

Wanted to drop zip_code thinking region column was enough.

Gemini:
If you drop zip_code and only keep region, you are telling the mathematical model that a 3-bedroom house in the ultra-wealthy coastal town of Knokke-Heist (Flanders) has the exact same baseline geographical value as a 3-bedroom house in a remote farming village in Limburg (also Flanders).

Wanted to drop property_type

Gemini:
A property in central Brussels costs vastly more than the exact same property in a rural village.

Me and Gemini:
Drop locality (redundant text), type_of_sale (zero variance), and subtype (too granular for now, property_type is enough).

Gemini:
Here is the professional, consolidated way to write this module. We do not overwrite df multiple times. We extract y, and we build X by dropping the target and the noise simultaneously.

In [53]:
#splitting zip code
#use first two digits to get specific info about municipalities/provinces

df['zip_code'] = df['zip_code'] // 100 

In [54]:
#define y target variable
y = df['price_eur']


#define X variable by dropping irrelevant columns (pure noise) and target
X = df.drop(columns=['locality', 'subtype', 'type_of_sale', 'price_eur'])


STEP 2 DATA SPLITTING

how to effectively choose an appropriate test_size value for any given dataset size:

 The Mechanics (Absolute vs. Percentage):
 
 Coders memorize "80/20" as a rule. Data Scientists look at the absolute number of rows.
 
 If you have $100$ rows: 20% is $20$ rows. Testing a model on 20 properties will not give you a reliable error metric. One outlier ruins the whole test.
 
 If you have $10,000,000$ rows: 20% is $2,000,000$ rows. An exam of 2 million questions is a massive waste of data. A 1% test size ($100,000$ rows) is more than enough to confidently evaluate the model, leaving 99% for training.
 
 Your Dataset: You have ~23,000 rows. 20% leaves you with ~18,400 rows to train on, and a test set of ~4,600 rows. 4,600 properties is a statistically significant sample size to evaluate a real estate market. Your choice of 0.2 was correct by accident. Now it is correct by design.

In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
 

(16690, 18)
(4173, 18)
(16690,)
(4173,)


STEP 3 The Preprocessing Pipeline

We have isolated X_train. We must now build the preprocessing engine. We will use scikit-learn's Pipeline and ColumnTransformer.

If you apply transformations (like filling NaNs or scaling) one column at a time manually, your code becomes unreadable and leaks data easily. A ColumnTransformer acts as a sorting facility. It takes your raw data, splits it into different "pipes" based on the data type, applies the correct math to each pipe, and stitches it back together.

Look at the columns remaining in your X_train. You must categorize them into two lists.

🔢 Numerical Features: Columns that contain continuous or discrete numbers where the magnitude matters (e.g., area, distances). These will need median imputation and scaling.

🔠 Categorical Features: Columns that contain text or numbers acting as categories (e.g., regions, zip codes, property type, binary flags like swimming pool). These need One-Hot Encoding.

In [56]:
X.info()


<class 'pandas.DataFrame'>
Index: 20863 entries, 1 to 23168
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   region                  20863 non-null  str    
 1   zip_code                20863 non-null  int64  
 2   property_type           20863 non-null  str    
 3   num_rooms               20863 non-null  int64  
 4   living_area_m2          20863 non-null  float64
 5   fully_equipped_kitchen  20863 non-null  int64  
 6   furnished               20863 non-null  int64  
 7   terrace                 20863 non-null  int64  
 8   terrace_area_m2         14230 non-null  float64
 9   garden                  20863 non-null  int64  
 10  garden_area_m2          13850 non-null  float64
 11  land_surface_m2         19337 non-null  float64
 12  num_facades             15991 non-null  float64
 13  swimming_pool           20863 non-null  int64  
 14  state_of_building       16512 non-null  str    
 15  n

In [57]:
#create num features and cat features lists for easier preprocessing
numeric_features = ['num_rooms', 
                      'living_area_m2', 
                      'terrace_area_m2', 
                      'garden_area_m2', 
                      'land_surface_m2', 
                      'num_facades', 
                      'num_bathrooms',  
                      'dist_train_km', 
                      'dist_bus_km']


nominal_features = ['region', 
                        'zip_code', 
                        'property_type', 
                        'fully_equipped_kitchen', 
                        'furnished', 
                        'terrace', 
                        'garden', 
                        'swimming_pool']

ordinal_features = ['state_of_building']

The Intuition: 

Think of ColumnTransformer as a sorting facility 🏭. Raw data enters, gets split onto a "Numeric Conveyor Belt" and a "Categorical Conveyor Belt," undergoes specific mathematical transformations, and is finally stitched back together into a single, clean mathematical matrix ready for the model.

Before we combine everything, we must build the individual conveyor belts. Let us start with the numeric pipeline.

For your Numerical list, we must execute two steps sequentially:

Impute missing values using the median (to avoid fractional rooms/facades).

Scale the values using StandardScaler so features with massive numbers (like land_surface_m2) do not mathematically dominate features with small numbers (like num_bathrooms).

The Numerical Pipeline

In [58]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import PolynomialFeatures

In [59]:
#create numeric_transformer pipeline to impute and scale numerical columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    #('polynomial', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler())
])

Step 3, Part 2: The Categorical Pipelines

The Architectural Trap:

You cannot send all of these down the exact same pipeline. Why? Because as we discussed, state_of_building is Ordinal (ranked), while region, zip_code, and property_type are Nominal (unranked).

If you put state_of_building into a OneHotEncoder, you destroy the rank. If you put region into an OrdinalEncoder, you create a fake mathematical hierarchy.

Therefore, you need two categorical pipelines:

nominal_transformer: For unranked text/categories.

ordinal_transformer: Specifically for state_of_building.

In [60]:

#create pipeline for nominal_categories to impute NANs with most_frequent value
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])


#create pipeline for ordinal state_of_building 
#to impute NANs with most_frequent value 
#and rank the states from lowest to highest
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories= [['To demolish', 
                                            'To restore', 
                                            'To be renovated', 
                                            'To renovate', 
                                            'Under construction',
                                            'Normal', 
                                            'Excellent', 
                                            'Fully renovated', 
                                            'New']],
                                handle_unknown='use_encoded_value',
                                unknown_value=-1))
                                
])

COLUMNTRANSFORMER

In [61]:
'''
ColumnTransformer takes each pipeline and directs into the specified relevant column features
remainder='drop' to ensure that if any garbage columns accidentally slipped into your dataset, 
they are automatically destroyed instead of crashing the model.
'''

from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('nom', nominal_transformer, nominal_features),
        ('ord', ordinal_transformer, ordinal_features)
    ],
    remainder='drop'
)

How to use it with your Model

Now, you just attach your chosen model (like Linear Regression) to the end of this preprocessor using a Pipeline.

In [62]:
from sklearn.linear_model import LinearRegression

model_pipeline_LR = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [63]:
# One single command to Clean, Encode, Scale, and Train!
model_pipeline_LR.fit(X_train, y_train)
model_pipeline_LR.score(X_train, y_train)

0.6408706882745163

In [64]:
model_pipeline_LR.score(X_test, y_test)

0.6397707167109602

## Permutation importance

In [65]:
from sklearn.inspection import permutation_importance

# Works for ANY model (LR, RFR, etc.)
perm_result = permutation_importance(model_pipeline_LR, X_test, y_test, n_repeats=10, random_state=42)

# Pull the scores
perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm_result.importances_mean,
    'std_dev': perm_result.importances_std
}).sort_values(by='importance', ascending=False)
perm_df


,feature,importance,std_dev
4,living_area_m2,0.450530,0.008966
0,region,0.224455,0.006658
1,zip_code,0.154772,0.004768
14,state_of_building,0.148440,0.003925
12,num_facades,0.035950,0.002752
15,num_bathrooms,0.032839,0.002075
2,property_type,0.017674,0.001613
3,num_rooms,0.017351,0.001165
11,land_surface_m2,0.008980,0.000518
7,terrace,0.006563,0.001769


It feels like a contradiction because we usually talk about them as opposites, but in reality, a model can be both bad at learning (Underfitting) AND bad at generalizing (Overfitting).

This is actually the "worst of both worlds," and it’s very common when using a simple Linear Regression on a complex project like Immo-Eliza.

Here is the plain English breakdown of how both are true at once:

1. Why it is Underfitting (The $0.59$ score)

    The $0.59$ Training score is your Ceiling.
    
    The Logic: Even when the model has seen all the answers, it only understands $59\%$ of what makes a house expensive. It is "too simple" to understand the complexity of the Belgian market. It’s trying to use a straight line to explain a market that has curves, local secrets, and weird exceptions.
    
    The Verdict: The model is not powerful enough.
2. Why it is Overfitting (The $0.59 \to 0.33$ gap)
    
    The fact that the score dropped significantly when it saw new data is the "Gap."
    
    The Logic: Even though it only understood $59\%$ of the data, it "cheated" to get some of that score. It likely found "weird coincidences" in your training data—for example, a specific Zip Code that happened to have three expensive houses. It assigned a huge value to that Zip Code.
    
    The Crash: When it got to the Test set and saw a cheap house in that same Zip Code, its logic "collapsed."The Verdict: The model is unstable.

### 📈 Linear Regression Model Progress Tracker

| Iteration | Key Modifications | Train $R^2$ | Test $R^2$ | Gap | Logic / Observation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | Baseline (Standard Scaling + OHE) | 0.59 | 0.33 | 0.26 | **High Variance:** Massive overfitting due to outliers and high-cardinality zones. |
| **2** | After IQR Outlier Removal | 0.64 | 0.63 | 0.01 | **Stabilized:** Removing extreme prices made the "formula" much more reliable. |
| **3** | Added Polynomial Features (Deg 2) | 0.66 | 0.66 | 0.00 | **Complexity:** Captured interactions between features. Slight boost. |

> **Notes:** > * The **Gap** is the difference between Train and Test. A smaller gap means a more "trustworthy" model.
> * Best model so far: **Iteration 3**

## TESTING WITH RANDOMFORESTREGRESSOR

No specific params yet in RandomForestRegressor to see raw behaviour

In [66]:
from sklearn.ensemble import RandomForestRegressor

model_pipeline_RFR = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [67]:
model_pipeline_RFR.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [68]:
model_pipeline_RFR.score(X_train, y_train)

0.9580710703149108

In [69]:
model_pipeline_RFR.score(X_test, y_test)

0.7190880267565014

Look at the metrics:

Your $R^2$ on the test set jumped from LR_score: $0.337$ to RFR_score: $0.725$. 

The council of trees 🌲 successfully mapped the non-linear synergies of the real estate market that the straight ruler failed to see. 

You are now explaining 72.5% of the variance in unseen property prices.

However, look at the gap:

Training Score: $0.966$ Testing Score: $0.725$

You have the tools to diagnose this. 

A training score of $0.966$ means the model almost perfectly memorized the answer key. 

A gap of roughly $0.24$ between training and testing is the mathematical signature of Overfitting 📉.

The Mechanics of the Overfit

Why did this happen? Because you ran the model using its default parameters.

By default, RandomForestRegressor sets max_depth=None.  

This means the algorithm is allowed to ask an infinite number of questions until it isolates every single property into its own distinct bucket. 

Instead of learning general market rules (e.g., "Houses in Zip 10 with 3 bedrooms are expensive"), it memorized the anomalies (e.g., "That one specific house with 3 facades, 2 bathrooms, and a 12m2 terrace that sold for a weird price").

## Feature importance in RFR

In [70]:
# 1. Get the feature names from the preprocessor (this can be tricky with OHE)
# If you have a simple setup, you can get names like this:
feature_names = model_pipeline_RFR.named_steps['preprocessor'].get_feature_names_out()

# 2. Get the importance scores from the random forest
importances = model_pipeline_RFR.named_steps['regressor'].feature_importances_

# 3. Put them in a nice DataFrame
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
importance_df = importance_df.sort_values(by='importance', ascending=False)

print(importance_df.head(10))

                   feature  importance
1      num__living_area_m2    0.322045
10    nom__region_Wallonia    0.108532
96  ord__state_of_building    0.094630
4     num__land_surface_m2    0.062014
8         num__dist_bus_km    0.047543
7       num__dist_train_km    0.047210
5         num__num_facades    0.045455
2     num__terrace_area_m2    0.037092
9     nom__region_Flanders    0.031141
6       num__num_bathrooms    0.026589


## Permutation importance in RFR

In [75]:
from sklearn.inspection import permutation_importance

# Works for ANY model (LR, RFR, etc.)
perm_result = permutation_importance(model_pipeline_RFR, X_test, y_test, scoring='r2', n_repeats=10, random_state=42)

# Pull the scores
perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm_result.importances_mean,
    'std_dev': perm_result.importances_std
}).sort_values(by='importance', ascending=False)
perm_df

,feature,importance,std_dev
4,living_area_m2,0.753100,0.015476
0,region,0.411206,0.013007
14,state_of_building,0.180774,0.006154
1,zip_code,0.094422,0.001951
11,land_surface_m2,0.061630,0.002413
12,num_facades,0.029450,0.001776
8,terrace_area_m2,0.023990,0.002410
15,num_bathrooms,0.021526,0.002586
16,dist_train_km,0.019632,0.001326
17,dist_bus_km,0.015765,0.001233


### Finetuning the RFR model with GridSearchCV (automated K Fold Cross Validation)

In [74]:
from sklearn.model_selection import GridSearchCV

#Define the parameters and their specific values to be tested
#We specify the model by calling it with regressor__
param_grid = {
    'regressor__n_estimators' : [100, 200],
    'regressor__max_depth' : [10, 15],
    'regressor__min_samples_split' : [10, 20, 30],
}

#Setup the GridSearch (using your existing pipeline)
RFR_grid_search = GridSearchCV(model_pipeline_RFR, param_grid, cv=5, scoring='r2', n_jobs=-1)


RFR_grid_search.fit(X_train, y_train)

print(f"Best Score: {RFR_grid_search.best_score_}")
print(f"Best Params: {RFR_grid_search.best_params_}")

#he grid search automatically saves the winning model inside an attribute called best_estimator_
best_model = RFR_grid_search.best_estimator_
print(f"Best Model Train Score {best_model.score(X_train, y_train)}")
print(f"Best Model Test Score {best_model.score(X_test, y_test)}")

Best Score: 0.6822421497256922
Best Params: {'regressor__max_depth': 15, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 200}
Best Model Train Score 0.8492078832404465
Best Model Test Score 0.705451069269001


### 📈 RandomForestRegressor Model Progress Tracker

| Iteration | Key Modifications | Train $R^2$ | Test $R^2$ | Gap | Logic / Observation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | Baseline raw, no params. No IQR outliers removal. Same Preprocessing as LR | 0.96 | 0.72 | 0.24 | Good but high gap: Overfitting, model memorized answer because max_depth=none. Model asks infinite num of questions |
| **2** | Baseline raw, no params. IQR outliers removal. | 0.95 | 0.71 | 0.23 | Same as first iteration, outliers removal dont influence model performance.|
| **3** | Removed Polynomial Features | Same | Same |  | RFR already sees on its own the kind of relationships that Poly features create |
| **4** | GridSearchCV: Best Params: {'regressor__max_depth': 30, 'regressor__min_samples_split': 3, 'regressor__n_estimators': 200} | Same | Same | | Model used max values offered. Didnt fix overfitting |
| **5** | GridSearchCV: Best Params: {'regressor__max_depth': 15, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 200}  | 0.84 | 0.70 | 0.14 | Enhancement in scores, less gap between train and test. Fixing Overfitting |


> **Notes:** > * The **Gap** is the difference between Train and Test. A smaller gap means a more "trustworthy" model.
> * Best model so far: 